# Módulo 00 — do Bronze ao Silver

Este caderno percorre o tratamento dos microdados de SRAG (SIVEP-Gripe) e mostra
a evidência de cada decisão. Ele **narra e verifica**; o tratamento em si vive em
`tools/srag_silver.py`, importado aqui — se o código estivesse duplicado no
caderno, os módulos seguintes divergiriam em silêncio.

A referência é o script oficial do Ministério da Saúde
([MIT, `gitlab.com/cgcovid/dados-publicos`](https://gitlab.com/cgcovid/dados-publicos)).
O que ele resolve é adotado; onde divergimos, o `DEVIATIONS` do módulo diz por quê.

In [1]:
%pip install -q pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pathlib
import subprocess
import sys
import urllib.request

import pandas as pd
import pyarrow.parquet as pq

# Os dados vêm do bucket público do Ministério — sem autenticação, sem upload.
# Se já existirem localmente (via tools/fetch_srag.sh), reaproveita.
S3 = "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SRAG"
DATA = pathlib.Path.home() / "Documents/srag-data"
if not DATA.exists():
    DATA = pathlib.Path("srag-data")
    DATA.mkdir(exist_ok=True)

def bronze(year):
    """Parquet de um ano, baixando do S3 se necessário."""
    name = f"INFLUD{year % 100:02d}-26-06-2025.parquet"
    local = DATA / name
    if not local.exists():
        urllib.request.urlretrieve(f"{S3}/{year}/{name}", local)
    return local

# O tratamento vem do repositório, não copiado para dentro do caderno.
REPO = pathlib.Path.cwd()
while REPO != REPO.parent and not (REPO / "tools" / "srag_silver.py").exists():
    REPO = REPO.parent
if not (REPO / "tools" / "srag_silver.py").exists():  # Colab
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/wbendinelli/interpretable-ml-lectures.git"], check=True)
    REPO = pathlib.Path("interpretable-ml-lectures")
sys.path.insert(0, str(REPO / "tools"))

import srag_silver as S

print(f"tratamento: {REPO / 'tools' / 'srag_silver.py'}")
print(f"dados:      {DATA}")

tratamento: /Users/wbendinelli/Documents/interpretable-ml-lectures/.claude/worktrees/code-audit-interpretability-252b08/tools/srag_silver.py
dados:      /Users/wbendinelli/Documents/srag-data


## 1. O defeito que motivou o resto

A regra parece óbvia: um checkbox de laboratório vale `1` quando marcado, vazio
quando não. Escrita como comparação literal contra `'1'`, ela funciona em cinco
anos e destrói o sexto.

In [3]:
col = "PCR_SARS2"
for year in (2019, 2020, 2021):
    s = pq.read_table(bronze(year), columns=[col]).to_pandas()[col]
    crus = sorted(s.dropna().astype(str).str.strip().unique())
    print(f"{year}: valores crus distintos = {crus}   preenchidos = {s.notna().sum():,}")

2019: valores crus distintos = ['1']   preenchidos = 12
2020: valores crus distintos = ['1.0']   preenchidos = 519,518
2021: valores crus distintos = ['1']   preenchidos = 703,702


Em 2020 o valor é `'1.0'`. Uma regra `== '1'` marca **zero** registros naquele
ano — e a contagem de linhas continua idêntica, então nenhuma verificação de
volume percebe.

A normalização precisa acontecer **antes** de inferir qualquer domínio, e precisa
cobrir mais que `.0`: o `RAIOX_RES` usa `2.0000000000` nos seis anos.

In [4]:
s = pq.read_table(bronze(2020), columns=[col]).to_pandas()[col]
ingenuo = (s.astype("string").str.strip() == "1").sum()
correto = (S.normalise(s, col) == "1").sum()
print(f"regra ingênua  == '1' : {ingenuo:>9,} registros")
print(f"após normalise()      : {correto:>9,} registros")
print(f"recuperados           : {correto - ingenuo:>9,}")

regra ingênua  == '1' :         0 registros
após normalise()      :   519,518 registros
recuperados           :   519,518


## 2. As três leituras de uma célula vazia

O vazio não quer dizer uma coisa só. Existe uma *variável-funil*
([Ribas et al., 2022](https://doi.org/10.1590/S1679-49742022000200004)):
quando `FATOR_RISC` não declara fator de risco, o sistema **desabilita** as treze
comorbidades. Esse vazio não é dado faltante — é campo que nunca foi oferecido.

In [5]:
cols = ["FATOR_RISC"] + S.COMORBIDITIES
d = pq.read_table(bronze(2021), columns=cols).to_pandas()
gate = S.normalise(d["FATOR_RISC"], "FATOR_RISC").isin(["1", "S"])
algum = pd.concat([S.normalise(d[c], c).notna() for c in S.COMORBIDITIES], axis=1).any(axis=1)

print(f"FATOR_RISC ausente  → alguma comorbidade preenchida: {100 * algum[~gate].mean():5.2f}%"
      f"  (n = {int((~gate).sum()):,})")
print(f"FATOR_RISC = Sim    → alguma comorbidade preenchida: {100 * algum[gate].mean():5.2f}%"
      f"  (n = {int(gate.sum()):,})")

FATOR_RISC ausente  → alguma comorbidade preenchida:  0.00%  (n = 771,391)
FATOR_RISC = Sim    → alguma comorbidade preenchida: 99.93%  (n = 974,281)


Zero por cento, sem uma exceção. O portão é absoluto — e é isso que separa
ausência estrutural de ausência real.

In [6]:
estado = S.missing_state(S.normalise(d["CARDIOPATI"], "CARDIOPATI"), gate)
vc = estado.value_counts()
for k, v in vc.items():
    print(f"  {k:<14} {v:>9,}   {100 * v / len(d):5.1f}%")
print(f"\nausência REAL de CARDIOPATI: {100 * (estado == 'ausente').sum() / gate.sum():.1f}%"
      f" de quem declarou fator de risco")

  nao_aplicavel    771,391    44.2%
  preenchido       759,175    43.5%
  ausente          207,240    11.9%
  ignorado           7,866     0.5%

ausência REAL de CARDIOPATI: 21.3% de quem declarou fator de risco


Contar todo vazio como ausência dá 56%. Separando os estados, a ausência real
é de 11,9% — o resto é campo desabilitado.

## 3. A definição de caso, e onde divergimos do Ministério

O script oficial define caso de SRAG por sintomas e **filtra** por ele. O
`criterio_1` aceita `HOSPITAL == 1` *ou* `EVOLUCAO == 2` — então pertencer à
coorte passa a depender do desfecho.

In [7]:
sil, quar = S.build(bronze(2021))
oficial = int(sil["caso_srag_ms"].sum())
nossa = int(sil["coorte_hospitalizado"].sum())
print(f"caso_srag_ms (critério oficial)      : {oficial:>9,}")
print(f"coorte_hospitalizado (sem o desfecho): {nossa:>9,}")
print(f"entram apenas por terem morrido      : {oficial - nossa:>9,}")

caso_srag_ms (critério oficial)      : 1,043,285
coorte_hospitalizado (sem o desfecho): 1,032,963
entram apenas por terem morrido      :    10,322


Para contar casos de vigilância isso é correto — é o que o informe semanal
precisa. Para treinar um modelo de desfecho é circular, então o Silver guarda os
dois: o critério oficial como coluna, e a coorte sem vazamento ao lado.

## 4. Nada é descartado

In [8]:
print(f"Bronze     : {len(sil) + len(quar):>9,}")
print(f"Silver     : {len(sil):>9,}")
print(f"Quarentena : {len(quar):>9,}   {quar['motivo'].iloc[0] if len(quar) else ''}")
assert len(sil) + len(quar) == pq.ParquetFile(bronze(2021)).metadata.num_rows
print("\ninvariante verificado: nenhuma linha perdida entre Bronze e Silver")

Bronze     : 1,745,672
Silver     : 1,745,669
Quarentena :         3   linha deslocada: coluna codificada contém data

invariante verificado: nenhuma linha perdida entre Bronze e Silver


As linhas em quarentena estão deslocadas em um campo: uma coluna codificada
passa a conter uma data, e tudo depois de `HOSPITAL` anda uma casa. Não são
quatorze anomalias de coluna independentes — é um defeito de linha, detectado
uma vez.

In [9]:
if len(quar):
    amostra = quar[["HOSPITAL", "DT_INTERNA", "UTI", "NM_UN_INTE"]].head(3)
    print(amostra.to_string(index=False))

  HOSPITAL DT_INTERNA        UTI NM_UN_INTE
2021-03-15         PR 2021-03-19          1
2021-09-04         SP 2021-09-04          1
      <NA>       <NA>       <NA>       <NA>


## O que o Silver entrega

| | |
|---|---|
| Granularidade | uma linha por notificação, inalterada |
| Universo | todas as linhas — em `silver` ou em `quarentena` |
| Valores | normalizados antes de qualquer inferência de domínio |
| Vazio | três estados distintos, nunca fundidos |
| Derivadas | `idade_anos`, `covid_caso`, `codeteccao`, `caso_srag_ms` |
| Decisões de tarefa | nenhuma — alvo, vazamento e filtro são do Gold |